In [2]:
import torch
from transformers import T5Tokenizer, Trainer ,T5ForConditionalGeneration, TrainingArguments

c:\Users\Administrator\Desktop\Tranformer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

In [5]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv("samsum-validation.csv")

In [6]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.shape

(14732, 3)

In [9]:
# random sampling

train_data = train_data.sample(n = 200 , random_state=42).reset_index(drop = True)
val_data = val_data.sample(n = 10 , random_state= 42).reset_index(drop=True)

In [10]:
train_data.shape

(200, 3)

# Data pre - procesing




In [4]:
import re

def clean_data(text):
    text = re.sub(r"\r\n" , " ", text) # line
    text = re.sub(r"\s+", " " , text) # spaces
    text = re.sub(r"<.*?>" , " ", text) # removel html tage
    text = text.strip().lower()
    return text

In [14]:
# Now Apply function

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary']  = train_data['summary'].apply(clean_data)


# valitadation
val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] =  val_data['summary'].apply(clean_data)

# Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [4]:
# raw data =. tokennized for fine-tuning

def tokenize(data):
    inputs = tokenizer(data['dialogue'], padding = 'max_length', max_length = 512 ,truncation = True)
    target = tokenizer(data['summary'] , padding  = 'max_length' , max_length = 150, truncation = True)
    
    inputs['labels'] = target['input_ids'] # token ids => add to input as lables
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize).tolist()
val_dataset = val_data.apply(tokenize).tolist()

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('gpu')
    
else:
    device = torch.device('cpu')
    
    
print('device', device)

In [ ]:
# Traning Arguments


training_Args = TrainingArguments(
    output_dir = "./result",
    
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    
    eval_strategy='epoch',
    save_strategy='epoch',
    
    warmup_steps=500
)

In [ ]:
# Training Steps performes

trainer = Trainer(
    
    model= model,
    args = training_Args,
    train_dataset = train_dataset,
    eval_dataset= val_dataset
)

In [ ]:
# train the model

trainer.train()

# Model load => fine_tune => save the model

In [5]:
model.save_pretrained("./save_summary_model")
tokenizer.save_pretrained("./save_summary_model")

NameError: name 'model' is not defined

In [3]:
model = T5ForConditionalGeneration.from_pretrained("./save/save_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./save/save_summary_model")


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2561.80it/s]


# Test the core logic for summarization

In [6]:
def summarize_dialogue(dialogue):
    
    dialogue = clean_data(dialogue) 
    
    # tokeniez
    
    inputs = tokenizer(
        dialogue,
        padding = 'max_length',
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
        
    )
    # generate the summary => token ids
    
    targets  = model.generate(
        input_ids = inputs['input_ids'],
        attention_mask = inputs['attention_mask'],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )
    
    
    #decoded our output
    
    summary = tokenizer.decode(targets[0], skip_special_tokens = True)
    return summary

In [7]:
summarize_dialogue("""
Artificial intelligence (AI) is the development of computer systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Driven by advancements in computing power and massive datasets, AI powers everything from virtual assistants to complex predictive analytics in global enterprise
""")

'Artificial intelligence (ai) is the development of computer systems capable of performing tasks that typically require human intelligence. ai powers everything from virtual assistants to complex predictive analytics in global enterprise.'